# Verify CTC dataset structure

This notebook verifies the raw-image and annotation structure of the three datasets selected for the Ultrack reproduction:

- `Fluo-N2DL-HeLa`
- `Fluo-N2DH-GOWT1`
- `Fluo-C3DL-MDA231`

Run the notebook on the cluster, where the datasets are already stored. It reads file names and lightweight TIFF metadata; it does not copy data or load complete sequences into memory.

CTC directory roles:

- `01/` and `02/`: raw microscopy images used as model input.
- `01_GT/` and `02_GT/`: ground-truth segmentation and tracking annotations used only for evaluation.
- `01_ST/` and `02_ST/`: silver-truth segmentations used only as references.
- `01_ERR_SEG/` and `02_ERR_SEG/`: intentionally imperfect segmentations supplied for challenge experiments; these are not ground truth and are not raw-image inputs.


## Imports and data location

Set `CTC_ROOT` in the shell before starting Jupyter. For example:

```bash
export CTC_ROOT=/path/to/shared/ctc
jupyter lab --no-browser
```

The active Python environment needs `pandas` and `tifffile`. Both are available in the Ultrack environment used for this project.


In [1]:
import sys

print(sys.executable)

/home/kshuaib/micromamba/envs/cell-tracking-ultrack/bin/python


In [ ]:
import os

os.environ["CTC_ROOT"] = "<PROJECT_DATA_ROOT>"

In [3]:
import re
from pathlib import Path

import pandas as pd
from tifffile import TiffFile

In [4]:
ctc_root_value = os.environ.get("CTC_ROOT")
if not ctc_root_value:
    raise RuntimeError("Set the CTC_ROOT environment variable before starting Jupyter.")

CTC_ROOT = Path(ctc_root_value).expanduser()
DATASETS = {
    "Fluo-N2DL-HeLa": 2,
    "Fluo-N2DH-GOWT1": 2,
    "Fluo-C3DL-MDA231": 3,
}
SEQUENCES = ("01", "02")

if not CTC_ROOT.is_dir():
    raise FileNotFoundError(f"CTC root does not exist: {CTC_ROOT}")

print(f"Reading datasets from: {CTC_ROOT}")

Reading datasets from: /net/spaces/clinic/CLS/DSC_CTC/ctc


## Helper functions

These helpers inspect file names and TIFF metadata. They do not read full image arrays.


In [12]:
INDEX_VALUES = re.compile(r"\d+")
SLICE_SEGMENTATION_INDEX_COUNT = 2
LINEAGE_FIELD_COUNT = 4


def file_indices(path):
    """Return all numeric indices encoded in a CTC file name."""
    return tuple(int(value) for value in INDEX_VALUES.findall(path.stem))


def frame_number(path):
    """Return the time index encoded in a CTC TIFF file name."""
    indices = file_indices(path)
    if not indices:
        raise ValueError(f"No frame number found in {path.name}")
    if (
        path.stem.startswith("man_seg_")
        and len(indices) == SLICE_SEGMENTATION_INDEX_COUNT
    ):
        return indices[0]
    return indices[-1]


def matching_files(directory, pattern):
    """Return naturally ordered files matching a CTC name pattern."""
    if not directory.is_dir():
        return []
    return sorted(directory.glob(pattern), key=file_indices)


def frame_range(files):
    """Return a compact first-to-last file-name range."""
    if not files:
        return None
    if len(files) == 1:
        return files[0].name
    return f"{files[0].name} to {files[-1].name}"


def tiff_metadata(path):
    """Read shape and dtype without loading a complete TIFF array."""
    with TiffFile(path) as tiff_file:
        series = tiff_file.series[0]
        return tuple(series.shape), str(series.dtype), len(tiff_file.pages)

## Check the expected directories

A standard CTC training dataset has two raw sequences, matching ground-truth folders, and usually silver-truth segmentation folders. This table makes missing directories visible rather than assuming they exist.


In [13]:
directory_rows = []

for dataset in DATASETS:
    dataset_root = CTC_ROOT / dataset
    for sequence in SEQUENCES:
        directory_rows.append(
            {
                "dataset": dataset,
                "sequence": sequence,
                "dataset root": dataset_root.is_dir(),
                "raw": (dataset_root / sequence).is_dir(),
                "GT/SEG": (dataset_root / f"{sequence}_GT/SEG").is_dir(),
                "GT/TRA": (dataset_root / f"{sequence}_GT/TRA").is_dir(),
                "ST/SEG": (dataset_root / f"{sequence}_ST/SEG").is_dir(),
                "ERR_SEG": (dataset_root / f"{sequence}_ERR_SEG").is_dir(),
            }
        )

directory_summary = pd.DataFrame(directory_rows)
directory_summary

,dataset,sequence,dataset root,raw,GT/SEG,GT/TRA,ST/SEG,ERR_SEG
0,Fluo-N2DL-HeLa,01,True,True,True,True,True,True
1,Fluo-N2DL-HeLa,02,True,True,True,True,True,True
2,Fluo-N2DH-GOWT1,01,True,True,True,True,True,True
3,Fluo-N2DH-GOWT1,02,True,True,True,True,True,True
4,Fluo-C3DL-MDA231,01,True,True,True,True,True,True
5,Fluo-C3DL-MDA231,02,True,True,True,True,True,True


Sequences 01 and 02 are two separate time-lapse microscopy movies from the same dataset—not two parts of one continuous movie.

They share the same imaging type and biological system, but may differ in:

- Cells and field of view
- Number of frames
- Cell density and divisions
- Image intensity or difficulty
- Annotation coverage

## Count files and record frame ranges

Ground-truth segmentation may cover fewer frames than the raw sequence. A lower `GT segmentation` count is therefore something to record, not automatically an error.


In [14]:
file_groups = {
    "raw": ("{sequence}", "t*.tif"),
    "GT segmentation": ("{sequence}_GT/SEG", "man_seg*.tif"),
    "GT tracking": ("{sequence}_GT/TRA", "man_track*.tif"),
    "ST segmentation": ("{sequence}_ST/SEG", "man_seg*.tif"),
}
file_rows = []
group_files = {}

for dataset in DATASETS:
    dataset_root = CTC_ROOT / dataset
    for sequence in SEQUENCES:
        for group, (folder_template, pattern) in file_groups.items():
            folder = dataset_root / folder_template.format(sequence=sequence)
            files = matching_files(folder, pattern)
            group_files[(dataset, sequence, group)] = files
            file_rows.append(
                {
                    "dataset": dataset,
                    "sequence": sequence,
                    "file group": group,
                    "count": len(files),
                    "file range": frame_range(files),
                }
            )

file_summary = pd.DataFrame(file_rows)
file_summary

,dataset,sequence,file group,count,file range
0,Fluo-N2DL-HeLa,01,raw,92,t000.tif to t091.tif
1,Fluo-N2DL-HeLa,01,GT segmentation,28,man_seg012.tif to man_seg088.tif
2,Fluo-N2DL-HeLa,01,GT tracking,92,man_track000.tif to man_track091.tif
3,Fluo-N2DL-HeLa,01,ST segmentation,92,man_seg000.tif to man_seg091.tif
4,Fluo-N2DL-HeLa,02,raw,92,t000.tif to t091.tif
5,Fluo-N2DL-HeLa,02,GT segmentation,8,man_seg023.tif to man_seg087.tif
6,Fluo-N2DL-HeLa,02,GT tracking,92,man_track000.tif to man_track091.tif
7,Fluo-N2DL-HeLa,02,ST segmentation,92,man_seg000.tif to man_seg091.tif
8,Fluo-N2DH-GOWT1,01,raw,92,t000.tif to t091.tif
9,Fluo-N2DH-GOWT1,01,GT segmentation,30,man_seg002.tif to man_seg047.tif


### Why the raw and ground-truth counts differ

Different counts between the ground truth and the raw data is expected as the counts are expected for these real microscopy datasets. 

**Raw**
The raw folders contain every time point in each movie, and the `GT/TRA` masks generally cover the complete sequence because tracking evaluation needs a lineage through time.

**GT Segmentation & ST Segmentation**

In contrast, `GT/SEG` is manually produced gold-standard segmentation and is intentionally sparse as expert annotation is labor-intensive, so only selected frames and representative cells are outlined precisely. 

The `ST/SEG` masks are computer-generated silver-standard annotations and therefore usually cover every raw frame.

**Frame count for 2D HeLa and GOWT1 datasets**

For the 2D HeLa and GOWT1 datasets, a name such as `man_seg012.tif` is the segmentation annotation for raw frame `t012.tif`. Missing `man_seg` numbers mean those raw frames do not have gold segmentation annotations; they do not indicate missing raw data.

**Frame count for 3D MDA231** 

The 3D MDA231 gold segmentation uses names such as `man_seg_001_007.tif`, where `001` is the time point and `007` is the annotated z-slice. Each of these files is one selected 2D plane, not an additional movie frame or a complete 3D volume. Consequently, its `GT segmentation` file count should not be compared directly with the number of raw 3D time points. The coverage check below extracts the time index separately from the z-slice index.

This follows the [Cell Tracking Challenge reference-annotation description](https://celltrackingchallenge.net/annotations/): real datasets have complete or near-complete tracking markers, limited-coverage gold segmentation, and broader-coverage silver segmentation.


## Inspect one TIFF from each file group

For 2D datasets, a raw TIFF should usually have shape `(y, x)`. For `Fluo-C3DL-MDA231`, each raw TIFF should usually have shape `(z, y, x)`. Annotation masks should have the same spatial shape as the corresponding raw images.


In [17]:
metadata_rows = []

for dataset, expected_dimensions in DATASETS.items():
    for sequence in SEQUENCES:
        raw_files = group_files[(dataset, sequence, "raw")]
        raw_shape = None
        if raw_files:
            raw_shape, _, _ = tiff_metadata(raw_files[0])

        for group in file_groups:
            files = group_files[(dataset, sequence, group)]
            if not files:
                continue
            shape, dtype, page_count = tiff_metadata(files[0])
            is_slice_annotation = (
                group == "GT segmentation"
                and files[0].stem.startswith("man_seg_")
                and len(file_indices(files[0])) == SLICE_SEGMENTATION_INDEX_COUNT
            )
            expected_shape = (
                raw_shape[-2:] if raw_shape and is_slice_annotation else raw_shape
            )
            metadata_rows.append(
                {
                    "dataset": dataset,
                    "sequence": sequence,
                    "file group": group,
                    "sample file": files[0].name,
                    "shape": shape,
                    "dtype": dtype,
                    "TIFF pages": page_count,
                    "shape matches expected raw plane/volume": (
                        shape == expected_shape
                    ),
                    "expected raw dimensions": (
                        len(shape) == expected_dimensions if group == "raw" else None
                    ),
                }
            )

metadata_summary = pd.DataFrame(metadata_rows)
metadata_summary

,dataset,sequence,file group,sample file,shape,dtype,TIFF pages,shape matches expected raw plane/volume,expected raw dimensions
0,Fluo-N2DL-HeLa,01,raw,t000.tif,"(700, 1100)",uint16,1,True,True
1,Fluo-N2DL-HeLa,01,GT segmentation,man_seg012.tif,"(700, 1100)",uint16,1,True,None
2,Fluo-N2DL-HeLa,01,GT tracking,man_track000.tif,"(700, 1100)",uint16,1,True,None
3,Fluo-N2DL-HeLa,01,ST segmentation,man_seg000.tif,"(700, 1100)",uint16,1,True,None
4,Fluo-N2DL-HeLa,02,raw,t000.tif,"(700, 1100)",uint16,1,True,True
5,Fluo-N2DL-HeLa,02,GT segmentation,man_seg023.tif,"(700, 1100)",uint16,1,True,None
6,Fluo-N2DL-HeLa,02,GT tracking,man_track000.tif,"(700, 1100)",uint16,1,True,None
7,Fluo-N2DL-HeLa,02,ST segmentation,man_seg000.tif,"(700, 1100)",uint16,1,True,None
8,Fluo-N2DH-GOWT1,01,raw,t000.tif,"(1024, 1024)",uint8,1,True,True
9,Fluo-N2DH-GOWT1,01,GT segmentation,man_seg002.tif,"(1024, 1024)",uint16,1,True,None


### Interpreting the TIFF metadata

**HeLa Dataset**
- The two HeLa sequences contain 2D `(700, 1100)` images, 

**GOWT1 Datatset**
- The two GOWT1 sequences contain 2D `(1024, 1024)` images. 
- The raw GOWT1 images use `uint8`, while their annotation masks use `uint16`. 
- This difference is expected as raw intensity images and labeled masks store different information, and `uint16` permits many distinct positive object labels.

**MDA 231 Dataset**
- Each MDA231 raw time point is a 3D `(30, 512, 512)` volume stored as a 30-page TIFF. Its tracking and silver-segmentation masks are matching 3D volumes. 
- Each gold-segmentation file is instead one annotated `(512, 512)` z-plane stored as a single-page TIFF. It should match the final two dimensions of the raw volume, not the complete 3D shape. 
-  The check reports whether the file matches the expected raw plane or volume.


## Compare annotation frame numbers with raw frames

Every annotation TIFF should correspond to an existing raw frame. This check does not require every raw frame to have a segmentation annotation.


In [21]:
coverage_rows = []

for dataset in DATASETS:
    for sequence in SEQUENCES:
        raw_numbers = {
            frame_number(path) for path in group_files[(dataset, sequence, "raw")]
        }
        for group in ("GT segmentation", "GT tracking", "ST segmentation"):
            annotation_numbers = {
                frame_number(path) for path in group_files[(dataset, sequence, group)]
            }
            missing_raw_frames = sorted(annotation_numbers - raw_numbers)
            coverage_rows.append(
                {
                    "dataset": dataset,
                    "sequence": sequence,
                    "annotation": group,
                    "annotated frames": len(annotation_numbers),
                    "annotations without raw frame": missing_raw_frames,
                    "count": len(
                        group_files[(f"{dataset}", f"{sequence}", f"{group}")]
                    ),
                }
            )

coverage_summary = pd.DataFrame(coverage_rows)
coverage_summary

,dataset,sequence,annotation,annotated frames,annotations without raw frame,count
0,Fluo-N2DL-HeLa,01,GT segmentation,28,[],28
1,Fluo-N2DL-HeLa,01,GT tracking,92,[],92
2,Fluo-N2DL-HeLa,01,ST segmentation,92,[],92
3,Fluo-N2DL-HeLa,02,GT segmentation,8,[],8
4,Fluo-N2DL-HeLa,02,GT tracking,92,[],92
5,Fluo-N2DL-HeLa,02,ST segmentation,92,[],92
6,Fluo-N2DH-GOWT1,01,GT segmentation,30,[],30
7,Fluo-N2DH-GOWT1,01,GT tracking,92,[],92
8,Fluo-N2DH-GOWT1,01,ST segmentation,92,[],92
9,Fluo-N2DH-GOWT1,02,GT segmentation,20,[],20


The table above indicates that every gold-segmentation, gold-tracking, and silver-segmentation annotation in all three datasets maps to an existing raw time point; all `annotations without raw frame` lists are empty.

For MDA231:
- Sequence 01: 17 annotated z-slices across 10 distinct time points
- Sequence 02: 13 annotated z-slices across 11 distinct time points

Some time points therefore contain more than one annotated z-slice


## 7. Verify lineage files

Each `GT/TRA/man_track.txt` row has four integers: cell label, first frame, last frame, and parent label. A parent value of `0` indicates no annotated parent.


In [22]:
lineage_rows = []

for dataset in DATASETS:
    dataset_root = CTC_ROOT / dataset
    for sequence in SEQUENCES:
        lineage_path = dataset_root / f"{sequence}_GT/TRA/man_track.txt"
        lines = []
        if lineage_path.is_file():
            lines = [
                line.strip()
                for line in lineage_path.read_text().splitlines()
                if line.strip()
            ]
        valid_rows = all(len(line.split()) == LINEAGE_FIELD_COUNT for line in lines)
        lineage_rows.append(
            {
                "dataset": dataset,
                "sequence": sequence,
                "lineage file exists": lineage_path.is_file(),
                "lineage records": len(lines),
                "four fields per row": valid_rows if lines else None,
                "first record": lines[0] if lines else None,
            }
        )

lineage_summary = pd.DataFrame(lineage_rows)
lineage_summary

,dataset,sequence,lineage file exists,lineage records,four fields per row,first record
0,Fluo-N2DL-HeLa,01,True,265,True,1 0 30 0
1,Fluo-N2DL-HeLa,02,True,674,True,1 0 0 0
2,Fluo-N2DH-GOWT1,01,True,28,True,1 0 91 0
3,Fluo-N2DH-GOWT1,02,True,58,True,2 0 32 0
4,Fluo-C3DL-MDA231,01,True,33,True,1 0 11 0
5,Fluo-C3DL-MDA231,02,True,61,True,1 0 11 0


Each row follows: `L B E P`

- `L` is the track label. Pixels or voxels belonging to this track have value `L` in `man_trackT.tif`. The same label is retained from frame `B` through frame `E`. Labels are unique track identifiers but do not have to be consecutive.
- `B` and `E` are the first and last time points containing that track. Both endpoints are included.
- `P` is the label of the immediate parent or predecessor track. It links daughter tracks to their mother after a division and can also link a reappearing track after a temporal gap. `P = 0` means that no parent is defined

For example a row with `1 0 30 0` means track label 1 appears from frames 0–30 and has no recorded parent.

A division might look like:
`1 0 30 0` ->  `2 31 91 1` -> `3 31 91 1`

Here, tracks 2 and 3 are daughters of track 1. Parent links can also represent a track reappearing after a gap. Labels do not need to be consecutive, so GOWT1 starting with label 2 is valid.

## Conclusions

### Raw-image count, File range, Shape, and Dtype for each sequence;

#### `Fluo-N2DL-HeLa`

| Check | Sequence 01 | Sequence 02 |
| --- | --- | --- |
| Raw-image count | 92 | 92 |
| Raw-image filename range | `t000.tif`–`t091.tif` | `t000.tif`–`t091.tif` |
| Image shape `(y, x)` | `(700, 1100)` | `(700, 1100)` |
| Data type | `uint16` | `uint16` |
| `GT/SEG` files | 28 | 8 |
| `GT/TRA` image files | 92 | 92 |
| `GT/TRA/man_track.txt` | Present; 265 records | Present; 674 records |
| `ST/SEG` files | 92 | 92 |

Notes:

- Gold segmentation is sparse: its file count is lower than the raw-image
  count, while tracking truth and silver segmentation cover all 92 frames.
- All sampled images and masks are single-page `(700, 1100)` TIFFs with
  `uint16` data.

#### `Fluo-N2DH-GOWT1`

| Check | Sequence 01 | Sequence 02 |
| --- | --- | --- |
| Raw-image count | 92 | 92 |
| Raw-image filename range | `t000.tif`–`t091.tif` | `t000.tif`–`t091.tif` |
| Image shape `(y, x)` | `(1024, 1024)` | `(1024, 1024)` |
| Data type | `uint8` | `uint8` |
| `GT/SEG` files | 30 | 20 |
| `GT/TRA` image files | 92 | 92 |
| `GT/TRA/man_track.txt` | Present; 28 records | Present; 58 records |
| `ST/SEG` files | 92 | 92 |

Notes:

- Gold segmentation is sparse, while tracking truth and silver segmentation
  cover all 92 frames.
- Raw images are `uint8`; the sampled label masks are `uint16`, which supports
  distinct positive integer labels for many objects.

#### `Fluo-C3DL-MDA231`

| Check | Sequence 01 | Sequence 02 |
| --- | --- | --- |
| Raw-volume count | 12 | 12 |
| Raw-volume filename range | `t000.tif`–`t011.tif` | `t000.tif`–`t011.tif` |
| Volume shape `(z, y, x)` | `(30, 512, 512)` | `(30, 512, 512)` |
| Data type | `uint16` | `uint16` |
| Voxel scale | TODO | TODO |
| `GT/SEG` files | 17 annotated z-slices | 13 annotated z-slices |
| `GT/TRA` image files | 12 volumes | 12 volumes |
| `GT/TRA/man_track.txt` | Present; 33 records | Present; 61 records |
| `ST/SEG` files | 12 volumes | 12 volumes |

Notes:

- Each raw, tracking-truth, and silver-segmentation TIFF is a 30-page 3D
  volume.
- Gold-segmentation files have shape `(512, 512)` and one TIFF page because
  each file represents a selected z-slice. In `man_seg_TTT_ZZZ.tif`, `TTT` is
  the time point and `ZZZ` is the z-slice. Their file counts must not be
  interpreted as counts of full 3D frames.
- Sequence 01 has 17 gold-segmentation slice files covering
  10 distinct time points, and sequence 02 has 13 slice files covering 11
  distinct time points. 
  
  
### Annotation matching frames
Every gold-segmentation, gold-tracking, and silver-segmentation annotation in all three datasets maps to an existing raw time point; all `annotations without raw frame` lists are empty.


### Tracking File Check with four fields per row

Every `GT/TRA/man_track.txt` file exists and every non-empty row has the four required `L B E P` fields: track label, beginning frame, ending frame, and parent label.
The record counts for sequences 01/02 respectively are:
- HeLa: 265/674
- GOWT1: 28/58
- MDA231: 33/61

### Missing or unexpected folders

No missing or unexpected folders.